# Model Analysis — Explainable Phishing URL Detection

This notebook is for exploration and presentation (e.g. for a viva), not for
producing the deployed model — that happens in `src/train_model.py`, which
this notebook mirrors.

**Important:** the cells below are not pre-executed. Run them locally
(`jupyter notebook`) after installing `requirements.txt`, so every number
and plot you see comes from your own machine, not a canned example.


## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import load_clean_dataset, get_registrable_domain
from src.preprocessing import build_feature_matrix
from src.feature_extraction import FEATURE_NAMES
from src.config import RANDOM_SEED, TEST_SIZE

pd.set_option("display.max_columns", 40)


## 2. Load and clean the dataset

See `src/data_loader.py` for why only the `URL` and `label` columns are kept — every other column in the raw PhiUSIIL CSV is derived from webpage content, which is not available at prediction time.

In [ ]:
data = load_clean_dataset()
print(data.shape)
data.head()


In [ ]:
data["label"].value_counts().rename({0: "legitimate", 1: "phishing"}).plot(
    kind="bar", title="Class balance (1 = phishing)"
)
plt.ylabel("count")
plt.show()


## 3. Feature extraction

Recomputes the 28 lexical features from the raw URL text using exactly the same function the live app uses (`src/feature_extraction.py`), so there is no train/predict mismatch.

In [ ]:
X = build_feature_matrix(data["url"])
y = data.loc[X.index, "label"]
print(X.shape)
X.describe().T


### Quick look: do phishing and legitimate URLs differ on a few raw features?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["url_length", "num_subdomains", "suspicious_keyword_count"]):
    for label, name in [(0, "legitimate"), (1, "phishing")]:
        ax.hist(X.loc[y == label, col], bins=30, alpha=0.5, label=name, density=True)
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()


## 4. Train/test split, grouped by domain

A plain random split can leak: the same domain (with near-identical URLs) could appear in both train and test, inflating test accuracy. Grouping by registrable domain avoids this.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups = data.loc[X.index, "url"].apply(get_registrable_domain)
splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(f"Train: {len(X_train)}  Test: {len(X_test)}")
print("Any domain overlap between train and test?",
      bool(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))


## 5. Compare candidate models

Same comparison `src/train_model.py` runs (3-fold CV, F1 on the phishing class, on a stratified subsample for speed). See the module docstring in `src/train_model.py` for why a subsample is used here and the *final* model is always refit on the full training set.

In [ ]:
from src.train_model import compare_models

comparison = compare_models(X_train, y_train)
pd.DataFrame(comparison).T[["cv_f1_mean", "cv_f1_std", "seconds"]]


## 6. Fit the selected model on the full training set and evaluate once on the test set

In [ ]:
from src.train_model import _candidate_models
from src.evaluation import compute_metrics, plot_confusion_matrix

best_name = max(comparison, key=lambda k: comparison[k]["cv_f1_mean"])
print("Selected:", best_name)

pipeline = _candidate_models()[best_name]
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]
metrics = compute_metrics(y_test, y_pred, y_proba)
metrics


In [ ]:
plot_confusion_matrix(np.array(metrics["confusion_matrix"]), "notebook_confusion_matrix.png",
                       title=f"Confusion Matrix - {best_name}")
plt.imshow(plt.imread("notebook_confusion_matrix.png"))
plt.axis("off")
plt.show()


## 7. Probability calibration check

Tree ensembles are not always well-calibrated: a predicted probability of 0.8 does not necessarily mean the model is right 80% of the time. This matters because the risk-level thresholds in `src/config.py` are defined directly on the predicted probability.

In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10)
plt.plot(prob_pred, prob_true, marker="o", label=best_name)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="perfectly calibrated")
plt.xlabel("Mean predicted probability")
plt.ylabel("Fraction of actual positives")
plt.title("Calibration curve")
plt.legend()
plt.show()


## 8. Explainability sanity-check

Run SHAP on a handful of test-set rows and eyeball whether the top features make intuitive sense.

In [ ]:
from src.explainability import explain_prediction, build_plain_language_summary

sample_row = X_test.iloc[[0]]
explanation = explain_prediction(pipeline, best_name, sample_row)
print("Method:", explanation["method"])
for line in build_plain_language_summary(explanation):
    print("-", line)


## 9. Conclusion

Summarise, in your own words once you've run the cells above:
- Which model was selected and why (referring to the actual CV table above)
- The final test-set precision/recall trade-off and why it matters for phishing detection
- Anything the calibration curve revealed about trusting the raw probability
- One limitation you'd highlight in a viva